In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [5]:
from src.recommendation_system.constants import CONFIG_PATH
from src.recommendation_system.utils.common import read_yaml , create_dir
from dataclasses import dataclass
from src.recommendation_system.logging import logger
import pickle

In [13]:

with open(r'artifacts\feature_engineering\featured_data\featured_data.pkl', 'rb') as f:
    sim_matrix = pickle.load(f)

print((sim_matrix.product_name_clean[0]))

men wonder sports running shoes


In [16]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class model_building_path:
    sim_matrix_path:  Path    
    featured_df_path: Path    
    model_path:       Path    


In [17]:
class Congif_manager:

    def __init__(self, config =CONFIG_PATH):
        self.config = read_yaml(config)

        create_dir([self.config.artifacts_root])

    def get_model_building(self) -> model_building_path:
        
        config = self.config.model_building

        create_dir([config.model_path])

        model_building_config = model_building_path(
            sim_matrix_path  = (config.sim_matrix_path),
            featured_df_path = (config.featured_df_path),
            model_path       = (config.model_path)
        )
        return model_building_config        

In [18]:
class model_building_config:

    def __init__(self, config) -> None:
        self.config = config
        self.sim_matrix = None
        self.df = None

    def load_artifacts(self):
        try:
            logger.info("=" * 20 + " Loading Artifacts STARTED " + "=" * 20)

            with open(self.config.sim_matrix_path, 'rb') as f:
                self.sim_matrix = pickle.load(f)
            with open(self.config.featured_df_path, 'rb') as f:
                self.df = pickle.load(f)

            logger.info("sim_matrix shape : %s", str(self.sim_matrix.shape))
            logger.info("df shape         : %s", str(self.df.shape))
            logger.info("df columns       : %s", list(self.df.columns))
            logger.info("=" * 20 + " Loading Artifacts COMPLETED " + "=" * 20)

            return self.sim_matrix, self.df

        except Exception as e:
            logger.error("Load artifacts FAILED - %s", str(e))
            raise e

    def recommendation_engine_1(self, product_id, n=10):
        # similarity based recommendations
        try:
            scores  = self.sim_matrix[product_id]
            top_idx = scores.argsort()[::-1][1:n+1]
            top_scores = scores[top_idx]

            results = self.df.iloc[top_idx][[
                'product_name_clean',
                'price',
                'rating',
                'aesthetic',
                'score',
            ]].copy()

            results['similarity'] = top_scores

            logger.info("[Engine 1] Top %d similar products found", n)
            logger.info("=" * 20 + " Engine 1 COMPLETED " + "=" * 20)

            return results, top_idx

        except Exception as e:
            logger.error("[Engine 1] FAILED - %s", str(e))
            raise e

    def recommendation_engine_2(self, aesthetic, n=10):
        # aesthetic/category based recommendations
        try:
            filtered = self.df[self.df['aesthetic'] == aesthetic]

            top_products = filtered.sort_values(
                'score', ascending=False
            ).head(n)

            logger.info("[Engine 2] Total in aesthetic : %d", len(filtered))
            logger.info("[Engine 2] Top %d products found", n)
            logger.info("=" * 20 + " Engine 2 COMPLETED " + "=" * 20)

            return top_products

        except Exception as e:
            logger.error("[Engine 2] FAILED - %s", str(e))
            raise e

    def initiate_recommendation(self, product_name, top_k=5):
        try:
            logger.info("=" * 20 + " Recommendation STARTED " + "=" * 20)

            # load artifacts
            self.sim_matrix, self.df = self.load_artifacts()

            # validate product exists
            if product_name not in self.df["product_name_clean"].values:
                raise ValueError(f"Product '{product_name}' not found")

            # get product_id and aesthetic
            product_id = self.df[self.df["product_name_clean"] == product_name].index[0]
            aesthetic  = self.df.loc[product_id, "aesthetic"]

            logger.info("Input product : %s", product_name)
            logger.info("Product ID    : %d", product_id)
            logger.info("Aesthetic     : %s", aesthetic)

            # engine 1 — similarity based
            recs_sim, top_idx = self.recommendation_engine_1(
                product_id=product_id,
                n=top_k
            )

            # engine 2 — aesthetic based
            recs_aesthetic = self.recommendation_engine_2(
                aesthetic=aesthetic,
                n=top_k
            )

            logger.info("=" * 20 + " Recommendation COMPLETED " + "=" * 20)

            return recs_sim, recs_aesthetic

        except Exception as e:
            logger.error("Recommendation FAILED - %s", str(e))
            raise e

In [20]:
con = Congif_manager()
model_feature = con.get_model_building()
model_feature = model_building_config(model_feature)
df = model_feature.initiate_recommendation('men wonder sports running shoes',top_k=10)


[2026-03-05 00:19:49,362: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-05 00:19:49,365: INFO: common: created directory at: artifacts]
[2026-03-05 00:19:49,366: INFO: common: created directory at: artifacts/model_building/model/]
[2026-03-05 00:19:49,368: INFO: 2908478837: ==================== Recommendation STARTED ====================]
[2026-03-05 00:19:49,369: INFO: 2908478837: ==================== Loading Artifacts STARTED ====================]
[2026-03-05 00:19:56,967: INFO: 2908478837: sim_matrix shape : (11951, 11951)]
[2026-03-05 00:19:56,967: INFO: 2908478837: df shape         : (11951, 16)]
[2026-03-05 00:19:56,967: INFO: 2908478837: df columns       : ['asin', 'product_name', 'price', 'rating', 'review_count', 'discount', 'image_url', 'product_link', 'aesthetic', 'is_new_product', 'product_name_clean', 'url_title', 'rich_text', 'rating_norm', 'review_norm', 'score']]
[2026-03-05 00:19:56,978: INFO: 2908478837: ==================== Loading Artifac

In [48]:
print(df["product_name"].head(20).tolist())


['mens half sleeve round neck cottonblend graphic print oversized tshirt man', 'half sleeve oversized cottonblend round neck drop shoulder printed mens tshirt color black', 'men half sleeve round neck graphic font printed cottonblend oversized tshirts colour orange', 'casual half sleeve cottonblend printed round neck drop shoulder oversized tshirt man color brown', 'oversized full sleeve cottonblend graphic printed round neck drop shoulder tshirt man color mustard', 'half sleeve oversized tshirt men round neck cottonblend drop shoulder printed tshirt black color', 'half sleeve cottonblend graphic print round neck drop shoulder oversized dailywear tshirt man color aqua blue', 'mens cotton graphic print oversized fit tshirt', 'pure cotton oversized loose baggy fit drop shoulder round neck cool front chest typographic logo printed raglan sleeve tshirt men colors white black swanwhite beige grey', 'heavy gsm pure cotton mens graphic print oversized fit half sleeve round neck cotton acid wa

In [23]:
import pickle
import pandas as pd

# load dataframe
df = pickle.load(open("artifacts/feature_engineering/featured_data/featured_data.pkl", 'rb'))

# load similarity matrix
sim_matrix = pickle.load(open("artifacts/feature_engineering/model/sim_matrix.pkl", 'rb'))

# print outputs
print("df shape        :", df.shape)
print("sim_matrix shape:", type(sim_matrix))
print("columns         :", list(df.columns))
print("aesthetics      :", df['aesthetic'].unique())
print("\nSample data:")
print(df.head(3))


df shape        : (13813, 14)
sim_matrix shape: <class 'numpy.ndarray'>
columns         : ['asin', 'product_name', 'price', 'rating', 'review_count', 'discount', 'image_url', 'product_link', 'aesthetic', 'is_new_product', 'style_text_v1', 'rating_norm', 'review_norm', 'score']
aesthetics      : ['streetwear' 'college_casual' 'old_money' 'date_clean' 'gym_athleisure'
 'y2k_party' 'office_smart']

Sample data:
         asin                                       product_name     price  \
0  B0FHDHSFYY  mens half sleeve round neck cottonblend graphi...  5.659482   
1  B0FK5PK4VG  half sleeve oversized cottonblend round neck d...  5.700444   
2  B0F99NP71S  men half sleeve round neck graphic font printe...  5.659482   

   rating  review_count  discount  \
0     4.0      6.102559        74   
1     4.0      6.006353        73   
2     4.2      5.375278        74   

                                           image_url  \
0  https://m.media-amazon.com/images/I/61WYx598KK...   
1  https://m.m

In [32]:
# pick first product
product_name = df['product_name'].iloc[296]
print("Input product:", product_name)

# get index
idx = df[df['product_name'] == product_name].index[0]
print("Index:", idx)

# get similarity scores
sim_scores = list(enumerate(sim_matrix[idx]))

# sort high to low
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

# exclude itself, take top 10
sim_scores = sim_scores[1:11]
indices    = [i[0] for i in sim_scores]
scores     = [round(i[1], 4) for i in sim_scores]

# get recommendations
result = df.iloc[indices][['product_name', 'aesthetic', 'price', 'rating', 'score']].copy()
result['similarity_score'] = scores

print("\nRecommendations:")
print(result.to_string())

Input product: loose fit track pants menlycra blend baggy pantsmens stylish baggy track pant
Index: 296

Recommendations:
                                                                                                                        product_name       aesthetic     price  rating     score  similarity_score
8958                                                   loose fit track pants menlycra blend baggy pantsmens stylish baggy track pant  gym_athleisure  6.061457     3.8  0.582998            1.0000
2986                                    jump cutsloose fit track pants menlycra blend baggy pantsmens stylish baggy track pant black  college_casual  5.771441     3.9  0.558000            0.9526
493    loose fit track pant men trouser pant men track pant men loose fit baggy track pant men track pant men men stylish track pant      streetwear  5.940171     3.8  0.579181            0.3846
2919   loose fit track pant men trouser pant men track pant men loose fit baggy track pant men tra

In [33]:
# after getting recommendations
# drop duplicate product names
# keep highest scored one

result = result.drop_duplicates(
    subset=['product_name'], 
    keep='first'
)

print("\nAfter removing duplicates:")
print(result.to_string())


After removing duplicates:
                                                                                                                        product_name       aesthetic     price  rating     score  similarity_score
8958                                                   loose fit track pants menlycra blend baggy pantsmens stylish baggy track pant  gym_athleisure  6.061457     3.8  0.582998            1.0000
2986                                    jump cutsloose fit track pants menlycra blend baggy pantsmens stylish baggy track pant black  college_casual  5.771441     3.9  0.558000            0.9526
493    loose fit track pant men trouser pant men track pant men loose fit baggy track pant men track pant men men stylish track pant      streetwear  5.940171     3.8  0.579181            0.3846
11115                                                       mens loose mid rise track pant trouser pant fit baggy stylish track pant       y2k_party  6.551080     4.5  0.603414            0.36

In [35]:
# input product aesthetic
true_aesthetic = df[df['product_name'] == product_name]['aesthetic'].values[0]
print("True aesthetic:", true_aesthetic)

# how many recommendations match
matching = (result['aesthetic'] == true_aesthetic).sum()
print("Matching aesthetic:", matching)

# precision@k
top_k     = len(result)
precision = matching / top_k
print(f"Precision@{top_k}:", round(precision, 4))

# total products with same aesthetic in df
total_relevant = (df['aesthetic'] == true_aesthetic).sum()
print("Total relevant in df:", total_relevant)

# recall@k
recall = matching / total_relevant
print(f"Recall@{top_k}:", round(recall, 4))


True aesthetic: streetwear
Matching aesthetic: 3
Precision@8: 0.375
Total relevant in df: 1927
Recall@8: 0.0016


In [36]:
print(df[['product_name', 'style_text_v1', 'aesthetic']].head(5).to_string())

                                                                                          product_name                                                      style_text_v1   aesthetic
0                           mens half sleeve round neck cottonblend graphic print oversized tshirt man                                cottonblend graphic print oversized  streetwear
1           half sleeve oversized cottonblend round neck drop shoulder printed mens tshirt color black            oversized cottonblend drop shoulder printed color black  streetwear
2          men half sleeve round neck graphic font printed cottonblend oversized tshirts colour orange   graphic font printed cottonblend oversized tshirts colour orange  streetwear
3     casual half sleeve cottonblend printed round neck drop shoulder oversized tshirt man color brown     casual cottonblend printed drop shoulder oversized color brown  streetwear
4  oversized full sleeve cottonblend graphic printed round neck drop shoulder tshirt man c